# Lecture 07 exercises — Forecasting with past information

Run the cells in order. Data loading, feature construction, and chronological boundaries are provided so you can focus on the forecast design. Try each prompt before opening `solutions.ipynb`. The code cells are runnable as written.

## 1. Information at the origin

In your own words, explain why $\mathbb E[Y_{t+1}\mid\mathcal F_t]$ is measurable with respect to information available at $t$. Name one feature in our table that is known for tomorrow and one value that is not.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_file(filename):
    """Search this directory and its parents for a file under data/."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / "data" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find data/{filename} from {Path.cwd()}")


sales_data = pd.read_csv(find_data_file("daily_sales.csv"), parse_dates=["date"])
sales_data = sales_data.sort_values("date").reset_index(drop=True)
assert sales_data["date"].is_monotonic_increasing and sales_data["date"].is_unique
sales_data.head()

In [ ]:
def make_forecasting_features(data):
    """Build rolling one-step predictors using only observations before each date."""
    frame = data[["date", "sales"]].copy().sort_values("date").reset_index(drop=True)
    day = frame["date"].dt.dayofweek.to_numpy()
    frame["weekday_sin"] = np.sin(2 * np.pi * day / 7)
    frame["weekday_cos"] = np.cos(2 * np.pi * day / 7)
    frame["year_day_sin"] = np.sin(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    frame["year_day_cos"] = np.cos(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    for lag in (1, 2, 7, 14, 28):
        frame[f"lag_{lag}"] = frame["sales"].shift(lag)
    frame["mean_7_before"] = frame["sales"].shift(1).rolling(7).mean()
    frame["mean_28_before"] = frame["sales"].shift(1).rolling(28).mean()
    return frame.dropna().reset_index(drop=True)


feature_data = make_forecasting_features(sales_data)
feature_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
feature_data[["date", "sales", "lag_1", "lag_7", "mean_7_before", "weekday_sin"]].head()

## 2. Check the lag window

Complete the written explanation: why must the rolling mean use `shift(1)` before `rolling(7)`? The assertions below give you a concrete boundary to inspect.

In [ ]:
row = feature_data.iloc[100]
expected_lag_7 = sales_data.loc[28 + 100 - 7, "sales"]
expected_mean = sales_data.loc[121:127, "sales"].mean()
assert row["lag_7"] == expected_lag_7
assert np.isclose(row["mean_7_before"], expected_mean)
print(
    "For",
    row["date"].date(),
    "lag_7 is",
    row["lag_7"],
    "and the preceding-week mean is",
    round(row["mean_7_before"], 2),
)

## Leakage property check

Change sales strictly after one origin. Its lag and rolling features at that earlier origin must stay the same.

In [ ]:
origin_raw_index = 200
origin_feature_index = origin_raw_index - 28
future_perturbation = sales_data.copy()
future_perturbation.loc[origin_raw_index + 1 :, "sales"] += 10000
perturbed_features = make_forecasting_features(future_perturbation)
predictor_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
np.testing.assert_allclose(
    feature_data.loc[origin_feature_index, predictor_columns].to_numpy(dtype=float),
    perturbed_features.loc[origin_feature_index, predictor_columns].to_numpy(
        dtype=float
    ),
)
print("Future perturbations leave origin features unchanged.")

## 3. Preserve chronological boundaries

Create the same 70%/15%/15% split and report the first and last date in each partition. Then explain why shuffling before this split changes the question being evaluated.

In [ ]:
n_rows = len(feature_data)
train_end = int(0.70 * n_rows)
validation_end = int(0.85 * n_rows)
train_data = feature_data.iloc[:train_end].copy()
validation_data = feature_data.iloc[train_end:validation_end].copy()
test_data = feature_data.iloc[validation_end:].copy()
pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "first": [
            part.date.min().date() for part in (train_data, validation_data, test_data)
        ],
        "last": [
            part.date.max().date() for part in (train_data, validation_data, test_data)
        ],
    }
)

## 4. Compare with a seasonal-naive rule

For each test origin, predict from the same weekday one week earlier using `lag_7`. Compute MAE and RMSE. Why is this baseline a meaningful competitor for data with weekly recurrence?

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = test_data["sales"].to_numpy()
seasonal_prediction = test_data["lag_7"].to_numpy()
seasonal_scores = {
    "MAE": mean_absolute_error(actual, seasonal_prediction),
    "RMSE": mean_squared_error(actual, seasonal_prediction) ** 0.5,
}
seasonal_scores

## 5. Fit and assess a tree model

Fit an `XGBRegressor` on training rows, select the number of trees with the validation block, and compare the selected model with the baseline on test. The test features are valid for rolling one-step evaluation because each origin uses only previously observed actuals. How would the feature updates differ for a fixed-origin multi-day forecast?

In [ ]:
from xgboost import XGBRegressor

feature_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
forecast_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.035,
    max_depth=3,
    min_child_weight=5,
    reg_lambda=5.0,
    early_stopping_rounds=25,
    eval_metric="rmse",
    random_state=17,
    n_jobs=2,
)
forecast_model.fit(
    train_data[feature_columns],
    train_data["sales"],
    eval_set=[(validation_data[feature_columns], validation_data["sales"])],
    verbose=False,
)
model_prediction = forecast_model.predict(test_data[feature_columns])
comparison = pd.DataFrame(
    {
        "seasonal naive": {
            "MAE": mean_absolute_error(actual, seasonal_prediction),
            "RMSE": mean_squared_error(actual, seasonal_prediction) ** 0.5,
        },
        "XGBoost": {
            "MAE": mean_absolute_error(actual, model_prediction),
            "RMSE": mean_squared_error(actual, model_prediction) ** 0.5,
        },
    }
).T
comparison

## 6. Inspect the forecast errors

Plot test predictions and residuals over time. Look for runs of same-sign residuals, which may reveal a misspecified trend or seasonality even if the average score looks reasonable.

In [ ]:
residuals = actual - model_prediction
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout="constrained")
axes[0].plot(test_data["date"], actual, label="actual", color="black")
axes[0].plot(test_data["date"], model_prediction, label="XGBoost")
axes[0].legend(frameon=False)
axes[1].scatter(test_data["date"], residuals, s=12)
axes[1].axhline(0, color="black", lw=1)
axes[1].set(ylabel="actual - predicted", xlabel="date")
fig.autofmt_xdate()
plt.show()

## 7. Evaluation design in one sentence

State the origin, horizon, parameter-update schedule, and observation-update schedule represented by the final test score. Then describe what must change to evaluate a fixed-origin 14-day forecast without seeing actuals from those 14 days.